<a href="https://colab.research.google.com/github/AirtonDomiciano/automatic_model_training/blob/main/notebooks/automatic_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

**Colab:** CPU é permitida. Uma GPU compatível com o PyTorch instalado (por exemplo, a T4 do Colab) é opcional e acelera a geração de áudios e o treinamento. Se não houver GPU, ou se a GPU existir mas não for suportada por esta versão do PyTorch, o fluxo continua na CPU. Depois de atualizar este notebook, use `Desconectar e excluir ambiente de execução` e execute as células em ordem. O treinamento usa um ambiente isolado Python 3.10; o Python do notebook não é substituído. A primeira instalação pode levar vários minutos. Se uma célula falhar, pare e copie o erro completo.


In [ ]:
# Isolated training environment: compatible with the legacy openWakeWord pipeline.
import io
import os
import shutil
import sys
import subprocess
import zipfile
from pathlib import Path

def run_command(args, **kwargs):
    print("+", " ".join(str(part) for part in args), flush=True)
    subprocess.run([str(part) for part in args], check=True, **kwargs)

def hey_pran_paths():
    if Path("/content").is_dir() and os.access("/content", os.W_OK):
        work = Path("/content")
    else:
        work = Path.home() / "hey_pran_work"
    return {
        "work": work,
        "env": work / "hey_pran_env",
        "python": work / "hey_pran_env" / "bin" / "python",
        "oww": work / "hey_pran_openwakeword_v060",
        "piper": work / "hey_pran_piper_generator",
    }

def uv_command(*args):
    uv_bin = shutil.which("uv")
    if uv_bin is None:
        run_command([sys.executable, "-m", "pip", "install", "uv==0.8.22"])
        uv_bin = shutil.which("uv")
    if uv_bin:
        return [uv_bin, *args]
    return [sys.executable, "-m", "uv", *args]

def imports_succeed(python_bin):
    probe = subprocess.run(
        [str(python_bin), "-c",
         "import torch, tensorflow, onnx, openwakeword, espeak_phonemizer, dp\n"
         "from onnx_tf.backend import prepare\n"
         "assert torch.__version__.startswith('2.1.2'), torch.__version__\n"
         "assert tensorflow.__version__.startswith('2.8.1'), tensorflow.__version__\n"],
        capture_output=True, text=True,
    )
    if probe.returncode != 0:
        print(probe.stdout)
        print(probe.stderr)
    return probe.returncode == 0

def clear_executable_stacks(env_dir):
    """Older manylinux wheels request an executable stack, which current kernels reject."""
    pt_gnu_stack = 0x6474E551
    cleared = 0
    for path in Path(env_dir).rglob("*.so"):
        data = bytearray(path.read_bytes())
        if data[:4] != b"\x7fELF" or data[4] != 2:
            continue
        phoff = int.from_bytes(data[32:40], "little")
        phentsize = int.from_bytes(data[54:56], "little")
        phnum = int.from_bytes(data[56:58], "little")
        for index in range(phnum):
            offset = phoff + index * phentsize
            if int.from_bytes(data[offset:offset + 4], "little") != pt_gnu_stack:
                continue
            flags = int.from_bytes(data[offset + 4:offset + 8], "little")
            if flags & 1:
                data[offset + 4:offset + 8] = (flags & ~1).to_bytes(4, "little")
                path.write_bytes(data)
                cleared += 1
            break
    return cleared

def phonemizer_checkpoint_ready(path):
    path = Path(path)
    if not path.exists():
        return False
    try:
        with zipfile.ZipFile(path) as archive:
            member = next(name for name in archive.namelist() if name.endswith("data.pkl"))
            pickle_data = archive.read(member)
    except (zipfile.BadZipFile, StopIteration):
        return False
    return b"cdeep_phonemizer." not in pickle_data and b"cdp.preprocessing.text" in pickle_data

def prepare_phonemizer_checkpoint(destination):
    destination = Path(destination)
    # Hugging Face hosts the ARPABET checkpoint under the newer deep_phonemizer module name.
    # openWakeWord 0.6.0 imports deep-phonemizer 0.0.19 as dp, so remap only the pickle paths.
    if not phonemizer_checkpoint_ready(destination):
        download(
            "https://huggingface.co/NRC-CNRC/en_us_cmudict_ipa_forward_g2p/resolve/main/en_us_cmudict_forward.pt",
            destination,
            expected_size=66724700,
        )
        buffer = io.BytesIO()
        with zipfile.ZipFile(destination) as source, zipfile.ZipFile(buffer, "w") as rewritten:
            for item in source.infolist():
                payload = source.read(item.filename)
                if item.filename.endswith("data.pkl"):
                    payload = payload.replace(b"deep_phonemizer.", b"dp.")
                rewritten.writestr(item, payload)
        destination.write_bytes(buffer.getvalue())
    if not phonemizer_checkpoint_ready(destination):
        raise RuntimeError("Phonemizer checkpoint is not loadable by deep-phonemizer 0.0.19")

def download(url, destination, expected_size=None):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if expected_size and destination.exists() and destination.stat().st_size == expected_size:
        print(f"Already downloaded: {destination}", flush=True)
        return
    run_command(["wget", "-c", "--tries=20", "--retry-connrefused", "-O", str(destination), url])
    if expected_size and destination.stat().st_size != expected_size:
        raise RuntimeError(
            f"{destination} has size {destination.stat().st_size}, expected {expected_size}"
        )

def ensure_espeak_ng(work_dir):
    if shutil.which("espeak-ng"):
        print("espeak-ng:", shutil.which("espeak-ng"), flush=True)
        return
    if subprocess.run(["sudo", "-n", "true"], capture_output=True).returncode == 0:
        run_command(["sudo", "-n", "apt-get", "update"])
        run_command(["sudo", "-n", "apt-get", "install", "-y", "espeak-ng"])
        return
    deb_dir = work_dir / ".debs"
    deb_dir.mkdir(parents=True, exist_ok=True)
    if not list(deb_dir.glob("espeak-ng_*.deb")):
        run_command(["apt-get", "download", "espeak-ng"], cwd=str(deb_dir))
    deb = sorted(deb_dir.glob("espeak-ng_*.deb"))[-1]
    prefix = work_dir / ".local"
    run_command(["dpkg-deb", "-x", str(deb), str(prefix)])
    os.environ["PATH"] = str(prefix / "usr" / "bin") + os.pathsep + os.environ["PATH"]
    if not shutil.which("espeak-ng"):
        raise RuntimeError("espeak-ng was extracted but is not executable")
    print("espeak-ng:", shutil.which("espeak-ng"), flush=True)

paths = hey_pran_paths()
WORK_DIR = paths["work"]
ENV_DIR = paths["env"]
TRAIN_PYTHON = paths["python"]
OWW_DIR = paths["oww"]
PIPER_DIR = paths["piper"]
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print("Working directory:", WORK_DIR, flush=True)
ensure_espeak_ng(WORK_DIR)

setup_marker = ENV_DIR / ".hey_pran_setup_ok"
imports_ok = TRAIN_PYTHON.exists() and imports_succeed(TRAIN_PYTHON)
if TRAIN_PYTHON.exists() and not imports_ok and clear_executable_stacks(ENV_DIR):
    print("Cleared executable-stack flags on installed wheels", flush=True)
    imports_ok = imports_succeed(TRAIN_PYTHON)

if not imports_ok:
    run_command(uv_command("python", "install", "3.10"))
    if not TRAIN_PYTHON.exists():
        run_command(uv_command("venv", "--python", "3.10", str(ENV_DIR)))
    if subprocess.run([str(TRAIN_PYTHON), "-m", "pip", "--version"], capture_output=True).returncode != 0:
        run_command(uv_command("pip", "install", "--python", str(TRAIN_PYTHON), "pip==24.0", "setuptools==69.5.1", "wheel"))
    else:
        run_command([str(TRAIN_PYTHON), "-m", "pip", "install", "pip==24.0", "setuptools==69.5.1", "wheel"])

    gpu_present = shutil.which("nvidia-smi") is not None and subprocess.run(
        ["nvidia-smi", "-L"], capture_output=True
    ).returncode == 0
    torch_index = "https://download.pytorch.org/whl/cu118" if gpu_present else "https://download.pytorch.org/whl/cpu"
    print("PyTorch index:", torch_index, flush=True)
    run_command([str(TRAIN_PYTHON), "-m", "pip", "install",
                 "torch==2.1.2", "torchaudio==2.1.2", "--index-url", torch_index])

    if not OWW_DIR.exists():
        run_command(["git", "clone", "--depth", "1", "--branch", "v0.6.0",
                     "https://github.com/dscripka/openWakeWord.git", str(OWW_DIR)])
    if not PIPER_DIR.exists():
        run_command(["git", "clone", "--depth", "1",
                     "https://github.com/dscripka/piper-sample-generator", str(PIPER_DIR)])

    requirements = [
        "numpy==1.23.5", "scipy==1.10.1", "protobuf==3.20.3",
        "tensorflow-cpu==2.8.1", "keras==2.8.0", "tensorflow-probability==0.16.0",
        "tensorflow-addons==0.16.1", "onnx==1.14.1",
        "onnxruntime==1.16.3", "tflite-runtime==2.14.0",
        "piper-phonemize==1.1.0", "espeak-phonemizer==1.3.1", "webrtcvad==2.0.10",
        "mutagen==1.47.0", "torchinfo==1.8.0", "torchmetrics==1.2.0",
        "speechbrain==0.5.14", "audiomentations==0.33.0",
        "torch-audiomentations==0.11.0", "acoustics==0.2.6",
        "pronouncing==0.2.0", "datasets==2.14.6", "pyarrow==12.0.1",
        "huggingface-hub==0.23.5", "fsspec==2023.9.2",
        "deep-phonemizer==0.0.19", "PyYAML==6.0.1", "soundfile==0.12.1",
    ]
    run_command([str(TRAIN_PYTHON), "-m", "pip", "install", *requirements])
    # onnx-tf pulls a full tensorflow build; the CPU build above is the one this pipeline uses.
    run_command([str(TRAIN_PYTHON), "-m", "pip", "install", "--no-deps", "onnx-tf==1.10.0"])
    run_command([str(TRAIN_PYTHON), "-m", "pip", "install", "-e", str(OWW_DIR)])
    run_command([str(TRAIN_PYTHON), "-m", "pip", "install",
                 "torch==2.1.2", "torchaudio==2.1.2", "--index-url", torch_index])
    run_command([str(TRAIN_PYTHON), "-m", "pip", "install",
                 "numpy==1.23.5", "protobuf==3.20.3", "tensorflow-cpu==2.8.1"])
    run_command([str(TRAIN_PYTHON), "-m", "pip", "check"])
    if clear_executable_stacks(ENV_DIR):
        print("Cleared executable-stack flags on installed wheels", flush=True)
    if not imports_succeed(TRAIN_PYTHON):
        raise RuntimeError("Isolated environment imports failed after installation")
    setup_marker.write_text("ok\n")

# generate_samples.py loads models/en-us-libritts-high.pt, next to the json shipped in the fork.
download(
    "https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt",
    PIPER_DIR / "models" / "en-us-libritts-high.pt",
    expected_size=255226835,
)
json_path = PIPER_DIR / "models" / "en-us-libritts-high.pt.json"
if not json_path.exists():
    raise RuntimeError(f"Missing Piper config {json_path}")

for filename in ("embedding_model.onnx", "embedding_model.tflite", "melspectrogram.onnx", "melspectrogram.tflite"):
    download(
        "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/" + filename,
        OWW_DIR / "openwakeword" / "resources" / "models" / filename,
    )

# The S3 bucket used by openWakeWord v0.6.0 now returns 403.
prepare_phonemizer_checkpoint(OWW_DIR / "openwakeword" / "resources" / "en_us_cmudict_forward.pt")

generator_file = PIPER_DIR / "generate_samples.py"
generator_source = generator_file.read_text()
original_load = "torch.load(model_path)"
cpu_load = 'torch.load(model_path, map_location="cpu")'
if original_load in generator_source:
    generator_file.write_text(generator_source.replace(original_load, cpu_load, 1))
elif cpu_load not in generator_source:
    raise RuntimeError("Piper checkpoint loader changed; review CPU compatibility before continuing.")

gpu_probe = subprocess.run(
    [str(TRAIN_PYTHON), "-c",
     "import torch\n"
     "print('torch', torch.__version__)\n"
     "print('cuda_build', torch.version.cuda)\n"
     "if not torch.cuda.is_available():\n"
     "    print('cpu_only')\n"
     "else:\n"
     "    major, minor = torch.cuda.get_device_capability()\n"
     "    arch = f'sm_{major}{minor}'\n"
     "    compiled = set(torch.cuda.get_arch_list())\n"
     "    print('device', torch.cuda.get_device_name(0))\n"
     "    print('arch', arch)\n"
     "    print('compiled', ' '.join(sorted(compiled)))\n"
     "    if arch not in compiled and f'compute_{major}{minor}' not in compiled:\n"
     "        print('gpu_unsupported')\n"
     "    else:\n"
     "        torch.zeros(1, device='cuda')\n"
     "        print('gpu_ok')\n"],
    capture_output=True, text=True, timeout=180,
)
print(gpu_probe.stdout, flush=True)
if gpu_probe.stderr:
    print(gpu_probe.stderr, flush=True)
site_packages = Path(subprocess.check_output(
    [str(TRAIN_PYTHON), "-c", "import site; print(site.getsitepackages()[0])"],
    text=True,
).strip())
sitecustomize = site_packages / "sitecustomize.py"
gpu_ok = gpu_probe.returncode == 0 and "gpu_ok" in gpu_probe.stdout
if gpu_ok:
    if sitecustomize.exists():
        sitecustomize.unlink()
    print("GPU will be used for generation and training.", flush=True)
else:
    sitecustomize.write_text(
        'import os\n'
        'os.environ["CUDA_VISIBLE_DEVICES"] = ""\n'
        'os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")\n'
    )
    print("GPU is absent or unsupported by this PyTorch build. Training continues on CPU.", flush=True)

print("Training environment ready:", TRAIN_PYTHON, flush=True)


In [ ]:
# All scientific imports and training run in the isolated Python, not the notebook interpreter.
import os
import subprocess
import sys
import textwrap
from pathlib import Path

def hey_pran_paths():
    if Path("/content").is_dir() and os.access("/content", os.W_OK):
        work = Path("/content")
    else:
        work = Path.home() / "hey_pran_work"
    return {
        "work": work,
        "python": work / "hey_pran_env" / "bin" / "python",
        "oww": work / "hey_pran_openwakeword_v060",
        "piper": work / "hey_pran_piper_generator",
    }

paths = hey_pran_paths()
WORK_DIR = paths["work"]
TRAIN_PYTHON = str(paths["python"])
OWW_DIR = paths["oww"]
PIPER_DIR = paths["piper"]
TRAIN_SCRIPT = str(OWW_DIR / "openwakeword" / "train.py")

def run_training_code(source):
    prelude = """import os, sys, uuid, yaml, subprocess
import numpy as np
import torch
import datasets
import scipy
from pathlib import Path
from tqdm import tqdm
"""
    env = os.environ.copy()
    env["HEY_PRAN_WORK_DIR"] = str(WORK_DIR)
    env["HEY_PRAN_OWW_DIR"] = str(OWW_DIR)
    env["HEY_PRAN_PIPER_DIR"] = str(PIPER_DIR)
    env["PYTHONUNBUFFERED"] = "1"
    result = subprocess.run(
        [TRAIN_PYTHON, "-u", "-c", prelude + textwrap.dedent(source)],
        cwd=str(WORK_DIR),
        env=env,
    )
    if result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, result.args)

run_training_code("""
import tensorflow as tf
import onnx
import espeak_phonemizer
import openwakeword
from dp.phonemizer import Phonemizer
from onnx_tf.backend import prepare
print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__, "GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU mode: continuing without GPU. Audio generation and training will take longer.")
phonemizer_path = str(Path(os.environ["HEY_PRAN_OWW_DIR"]) / "openwakeword" / "resources" / "en_us_cmudict_forward.pt")
phonemizer = Phonemizer.from_checkpoint(phonemizer_path)
print("Phonemes for pran:", phonemizer("pran", lang="en_us"))
""")


In [ ]:
import shutil
import subprocess
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])
else:
    print("nvidia-smi not found. Continuing on CPU.")


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
run_training_code("""
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = Path("./mit_rirs")
output_dir.mkdir(exist_ok=True)
marker = output_dir / ".complete"
if marker.exists() and any(output_dir.glob("*.wav")):
    print(f"MIT impulse responses already prepared ({len(list(output_dir.glob('*.wav')))} files)")
else:
    rir_dataset = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True
    )
    count = 0
    for row in tqdm(rir_dataset):
        name = row["audio"]["path"].split("/")[-1]
        if not name.lower().endswith(".wav"):
            name = f"{count:04d}.wav"
        scipy.io.wavfile.write(
            output_dir / name, 16000, (row["audio"]["array"] * 32767).astype(np.int16)
        )
        count += 1
    if count == 0:
        raise RuntimeError("MIT impulse response dataset returned no clips")
    marker.write_text(str(count) + "\\n")
    print("Saved", count, "impulse responses")
""")


In [ ]:
run_training_code("""
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# The current main branch was converted to parquet and no longer publishes the tar shards.
# This commit still contains bal_train09.tar.

audioset_dir = Path("audioset")
audioset_dir.mkdir(exist_ok=True)
fname = "bal_train09.tar"
tar_path = audioset_dir / fname
expected_tar_size = 968540160
link = (
    "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/"
    "cac0e7817034e82a07d61c048561fc3f62c35da3/data/" + fname
)
if not tar_path.exists() or tar_path.stat().st_size != expected_tar_size:
    subprocess.run(
        ["wget", "-c", "--tries", "20", "--retry-connrefused", "-O", str(tar_path), link],
        check=True,
    )
if tar_path.stat().st_size != expected_tar_size:
    raise RuntimeError(f"Unexpected AudioSet archive size: {tar_path.stat().st_size}")

flacs = list(Path("audioset/audio").glob("**/*.flac")) if Path("audioset/audio").exists() else []
if not flacs:
    subprocess.run(["tar", "-xf", fname], check=True, cwd="audioset")
    flacs = list(Path("audioset/audio").glob("**/*.flac"))
print("AudioSet flac files:", len(flacs))
if not flacs:
    raise RuntimeError("AudioSet archive did not contain flac files")

output_dir = Path("./audioset_16k")
output_dir.mkdir(exist_ok=True)
marker = output_dir / ".complete"
if marker.exists() and any(output_dir.glob("*.wav")):
    print(f"AudioSet 16 kHz wavs already prepared ({len(list(output_dir.glob('*.wav')))} files)")
else:
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(path) for path in flacs]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    count = 0
    for row in tqdm(audioset_dataset):
        name = Path(row["audio"]["path"]).name.replace(".flac", ".wav")
        scipy.io.wavfile.write(
            output_dir / name, 16000, (row["audio"]["array"] * 32767).astype(np.int16)
        )
        count += 1
    marker.write_text(str(count) + "\\n")
    print("Saved", count, "AudioSet clips")

# Free Music Archive dataset (https://github.com/mdeff/fma)
fma_dir = Path("./fma")
fma_dir.mkdir(exist_ok=True)
n_hours = 1  # use only 1 hour of clips for this example notebook
n_clips = n_hours * 3600 // 30  # FMA small clips are 30 seconds
fma_marker = fma_dir / ".complete"
if fma_marker.exists() and fma_marker.read_text().strip() == str(n_clips):
    print(f"FMA clips already prepared ({n_clips} files)")
else:
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
    for i in tqdm(range(n_clips)):
        row = next(fma_dataset)
        name = Path(row["audio"]["path"]).name
        for extension in (".mp3", ".wav", ".flac", ".ogg"):
            if name.lower().endswith(extension):
                name = name[: -len(extension)] + ".wav"
                break
        else:
            name = f"{i:04d}.wav"
        scipy.io.wavfile.write(
            fma_dir / name, 16000, (row["audio"]["array"] * 32767).astype(np.int16)
        )
    fma_marker.write_text(str(n_clips) + "\\n")
    print("Saved", n_clips, "FMA clips")
""")


In [ ]:
run_training_code("""
# Download pre-computed openWakeWord features for training and validation.
# Training uses memory-mapped arrays, so the 17 GB negative-feature file does not need to fit in RAM.

features = {
    "openwakeword_features_ACAV100M_2000_hrs_16bit.npy": (
        "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
        17280000128,
    ),
    "validation_set_features.npy": (
        "https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy",
        184836608,
    ),
}
for filename, (url, expected_size) in features.items():
    path = Path(filename)
    if path.exists() and path.stat().st_size == expected_size:
        print(f"Already downloaded: {filename}")
        continue
    subprocess.run(
        ["wget", "-c", "--tries", "20", "--retry-connrefused", "-O", filename, url],
        check=True,
    )
    if path.stat().st_size != expected_size:
        raise RuntimeError(f"{filename} has size {path.stat().st_size}, expected {expected_size}")
    print(f"Downloaded {filename}")
""")


# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey pran"
- We'll only generate 1,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [ ]:
run_training_code("""
config_path = Path(os.environ["HEY_PRAN_OWW_DIR"]) / "examples" / "custom_model.yml"
with open(config_path) as file:
    config = yaml.safe_load(file)
print(config)
""")


In [ ]:
run_training_code("""
config_path = Path(os.environ["HEY_PRAN_OWW_DIR"]) / "examples" / "custom_model.yml"
with open(config_path) as file:
    config = yaml.safe_load(file)

config["target_phrase"] = ["hey pran"]
config["model_name"] = config["target_phrase"][0].replace(" ", "_")
config["n_samples"] = 1000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ["./audioset_16k", "./fma"]
config["background_paths_duplication_rate"] = [1, 1]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
config["piper_sample_generator_path"] = os.environ["HEY_PRAN_PIPER_DIR"]
# train.py calls generate_samples() without a model argument, so the checkpoint must be
# the default file next to en-us-libritts-high.pt.json.
if not torch.cuda.is_available():
    # Keep negative-example generation at a non-zero batch (tts_batch_size // 7).
    config["tts_batch_size"] = 14
with open("my_model.yaml", "w") as file:
    yaml.dump(config, file)
print("Saved my_model.yaml for", config["target_phrase"])
print("samples", config["n_samples"], "steps", config["steps"], "tts_batch_size", config["tts_batch_size"])
print("GPU available:", torch.cuda.is_available())
""")


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [ ]:
subprocess.run(
    [TRAIN_PYTHON, "-u", TRAIN_SCRIPT, "--training_config", "my_model.yaml", "--generate_clips"],
    check=True,
    cwd=str(WORK_DIR),
)


In [ ]:
subprocess.run(
    [TRAIN_PYTHON, "-u", TRAIN_SCRIPT, "--training_config", "my_model.yaml", "--augment_clips"],
    check=True,
    cwd=str(WORK_DIR),
)


In [ ]:
subprocess.run(
    [TRAIN_PYTHON, "-u", TRAIN_SCRIPT, "--training_config", "my_model.yaml", "--train_model"],
    check=True,
    cwd=str(WORK_DIR),
)


In [ ]:
run_training_code("""
with open("my_model.yaml") as file:
    config = yaml.safe_load(file)

def convert_onnx_to_tflite(onnx_model_path, output_path):
    import logging
    import tempfile
    import onnx
    import tensorflow as tf
    from onnx_tf.backend import prepare

    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()
        logging.info(f"####\\nSaving tflite model to '{output_path}'")
        with open(output_path, "wb") as f:
            f.write(tflite_model)

onnx_path = f"my_custom_model/{config['model_name']}.onnx"
tflite_path = f"my_custom_model/{config['model_name']}.tflite"
if not os.path.exists(onnx_path):
    raise FileNotFoundError(onnx_path)
if not os.path.exists(tflite_path):
    convert_onnx_to_tflite(onnx_path, tflite_path)

import numpy as np
import onnxruntime as ort
import tensorflow as tf

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
onnx_input = session.get_inputs()[0]
onnx_output = session.get_outputs()[0]
shape = [1 if not isinstance(dim, int) else dim for dim in onnx_input.shape]
dummy = np.random.rand(*shape).astype(np.float32)
onnx_result = session.run([onnx_output.name], {onnx_input.name: dummy})[0]
print("ONNX", onnx_path, "input", onnx_input.name, shape, "output", getattr(onnx_result, "shape", None))

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
tflite_input = interpreter.get_input_details()[0]
tflite_output = interpreter.get_output_details()[0]
interpreter.set_tensor(tflite_input["index"], dummy.astype(tflite_input["dtype"]))
interpreter.invoke()
tflite_result = interpreter.get_tensor(tflite_output["index"])
print("TFLite", tflite_path, "input", tflite_input["shape"].tolist(), "output", tflite_result.shape)
print("ONNX and TFLite models loaded and ran inference.")
""")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!